# Фінальний проєкт: infectious_cases (PostgreSQL)

Дані: статистика інфекційних захворювань за країнами та роками. СУБД: локальний PostgreSQL 16 (pgAdmin), база `fp`, схема `pandemic`.

## Завдання 1. Завантаження даних

In [ ]:
%pip install -q psycopg2-binary sqlalchemy pandas

In [ ]:
import os
import getpass
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Пароль не зберігається у файлі: береться зі змінної середовища або вводиться вручну.
DB_URL = os.getenv("FP_DB_URL")
if not DB_URL:
    DB_URL = f"postgresql+psycopg2://postgres:{getpass.getpass('Пароль користувача postgres: ')}@localhost:5432/fp"

engine = create_engine(DB_URL, future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


def run(sql_text):
    """Виконує SQL (можна кілька команд). Якщо остання повертає рядки, віддає DataFrame."""
    with engine.begin() as conn:
        res = conn.exec_driver_sql(sql_text.replace("%", "%%"))
        if res.returns_rows:
            return pd.DataFrame(res.fetchall(), columns=list(res.keys()))

Створюємо схему `pandemic` і робимо її схемою за замовчуванням для всіх наступних запитів.

In [ ]:
run(r"""
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;
""")

In [ ]:
# Схему за замовчуванням задаємо на рівні бази, щоб вона діяла для кожного підключення з пулу
run("ALTER DATABASE " + engine.url.database + " SET search_path TO pandemic, public")
engine.dispose()
run("SHOW search_path")

Імпорт `infectious_cases.csv` у таблицю `infectious_cases` у схемі `pandemic`. Усі показники беремо як TEXT: у файлі є порожні та нечислові значення, тому спершу завантажуємо дані як є, а типи приводимо в запитах.

In [ ]:
CSV_PATH = os.getenv("INFECTIOUS_CSV") or r"C:\Users\PC\Downloads\infectious_cases.csv"

df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
df.columns = [c.strip().lower() for c in df.columns]
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df = df.replace("", None)

df.to_sql("infectious_cases", engine, schema="pandemic", if_exists="replace", index=False, chunksize=10_000)
print(CSV_PATH, df.shape)

Перегляд структури та вмісту даних.

In [ ]:
run(r"""
SELECT * FROM pandemic.infectious_cases LIMIT 10;
""")

In [ ]:
run(r"""
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT (entity, code)) AS n_entities FROM pandemic.infectious_cases;
""")

In [ ]:
run(r"""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'pandemic' AND table_name = 'infectious_cases'
ORDER BY ordinal_position;
""")

## Завдання 2. Нормалізація до 3НФ

Створюємо дві таблиці: `entities` з унікальними комбінаціями Entity і Code та `infectious_cases_norm` з показниками, де замість повторюваних Entity і Code стоїть `entity_id`.

In [ ]:
run(r"""
DROP TABLE IF EXISTS pandemic.infectious_cases_norm;
DROP TABLE IF EXISTS pandemic.entities;

CREATE TABLE pandemic.entities (
    id     SERIAL PRIMARY KEY,
    entity TEXT NOT NULL,
    code   TEXT,
    UNIQUE (entity, code)
);

INSERT INTO pandemic.entities (entity, code)
SELECT DISTINCT entity, code
FROM pandemic.infectious_cases
ORDER BY entity;
""")

In [ ]:
run(r"""
CREATE TABLE pandemic.infectious_cases_norm (
    id                   SERIAL PRIMARY KEY,
    entity_id            INT NOT NULL REFERENCES pandemic.entities (id),
    year                 INT,
    number_yaws          TEXT,
    polio_cases          TEXT,
    cases_guinea_worm    TEXT,
    number_rabies        TEXT,
    number_malaria       TEXT,
    number_hiv           TEXT,
    number_tuberculosis  TEXT,
    number_smallpox      TEXT,
    number_cholera_cases TEXT
);

INSERT INTO pandemic.infectious_cases_norm (
    entity_id, year, number_yaws, polio_cases, cases_guinea_worm,
    number_rabies, number_malaria, number_hiv, number_tuberculosis,
    number_smallpox, number_cholera_cases
)
SELECT
    e.id, ic.year, ic.number_yaws, ic.polio_cases, ic.cases_guinea_worm,
    ic.number_rabies, ic.number_malaria, ic.number_hiv, ic.number_tuberculosis,
    ic.number_smallpox, ic.number_cholera_cases
FROM pandemic.infectious_cases ic
JOIN pandemic.entities e
  ON e.entity = ic.entity
 AND e.code IS NOT DISTINCT FROM ic.code;
""")

Перевірка: кількість рядків у нормалізованій таблиці має збігатися з сирою.

In [ ]:
run(r"""
SELECT
    (SELECT COUNT(*) FROM pandemic.infectious_cases)      AS raw_rows,
    (SELECT COUNT(*) FROM pandemic.infectious_cases_norm) AS norm_rows,
    (SELECT COUNT(*) FROM pandemic.entities)              AS entities_rows;
""")

In [ ]:
# контроль: обидві таблиці існують і рядків стільки ж
chk = run("""
SELECT (SELECT COUNT(*) FROM pandemic.infectious_cases) AS raw_rows,
       (SELECT COUNT(*) FROM pandemic.infectious_cases_norm) AS norm_rows
""")
assert chk.loc[0, "raw_rows"] == chk.loc[0, "norm_rows"], "кількість рядків не збігається"
print("OK: нормалізація без втрат рядків")

In [ ]:
run(r"""
SELECT e.entity, e.code, n.year, n.number_rabies, n.number_malaria
FROM pandemic.infectious_cases_norm n
JOIN pandemic.entities e ON e.id = n.entity_id
ORDER BY n.id
LIMIT 10;
""")

## Завдання 3. Аналіз Number_rabies

Для кожної унікальної комбінації Entity і Code: середнє, мінімум, максимум і сума. Беремо лише числові значення (порожні та нечислові відсікає регулярний вираз). Сортуємо за середнім за спаданням, виводимо 10 рядків.

In [ ]:
run(r"""
SELECT
    e.id,
    e.entity,
    e.code,
    AVG(n.number_rabies::NUMERIC) AS avg_rabies,
    MIN(n.number_rabies::NUMERIC) AS min_rabies,
    MAX(n.number_rabies::NUMERIC) AS max_rabies,
    SUM(n.number_rabies::NUMERIC) AS sum_rabies
FROM pandemic.infectious_cases_norm n
JOIN pandemic.entities e ON e.id = n.entity_id
WHERE n.number_rabies ~ '^-?[0-9]+(\.[0-9]+)?$'
GROUP BY e.id, e.entity, e.code
ORDER BY avg_rabies DESC
LIMIT 10;
""")

## Завдання 4. Різниця в роках (вбудовані функції)

`MAKE_DATE` будує 1 січня відповідного року, `CURRENT_DATE` дає поточну дату, `AGE` разом з `EXTRACT(YEAR ...)` дає повну різницю в роках.

In [ ]:
run(r"""
SELECT
    n.id,
    e.entity,
    n.year,
    MAKE_DATE(n.year, 1, 1)                                             AS year_start,
    CURRENT_DATE                                                        AS today,
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(n.year, 1, 1)))::INT  AS year_diff
FROM pandemic.infectious_cases_norm n
JOIN pandemic.entities e ON e.id = n.entity_id
WHERE n.year IS NOT NULL
ORDER BY n.id
LIMIT 20;
""")

## Завдання 5. Власні функції

**Функція 1** `year_diff(p_year)` приймає рік і повертає різницю в роках між поточною датою та 1 січня цього року.

**Функція 2** `avg_cases_per_period(p_cases, p_divisor)` повертає середню кількість захворювань за період: кількість за рік поділена на дільник (12 для місяця, 4 для кварталу, 2 для півріччя). `NULLIF` захищає від ділення на нуль.

In [ ]:
run(r"""
CREATE OR REPLACE FUNCTION pandemic.year_diff(p_year INT)
RETURNS INT
LANGUAGE SQL
STABLE
AS $$
    SELECT EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INT;
$$;
""")

In [ ]:
run(r"""
CREATE OR REPLACE FUNCTION pandemic.avg_cases_per_period(p_cases NUMERIC, p_divisor NUMERIC)
RETURNS NUMERIC
LANGUAGE SQL
IMMUTABLE
AS $$
    SELECT p_cases / NULLIF(p_divisor, 0);
$$;
""")

Перевірка функцій на простих значеннях: `year_diff(1996)` і `avg_cases_per_period(120, 12 / 4 / 2)`.

In [ ]:
run(r"""
SELECT
    pandemic.year_diff(1996)                         AS years_since_1996,
    pandemic.avg_cases_per_period(120, 12)           AS per_month,
    pandemic.avg_cases_per_period(120, 4)            AS per_quarter,
    pandemic.avg_cases_per_period(120, 2)            AS per_half_year;
""")

Обидві функції в одному запиті, лише для записів із числовим значенням `Number_rabies`.

In [ ]:
run(r"""
SELECT
    e.entity,
    n.year,
    pandemic.year_diff(n.year)                                              AS years_ago,
    n.number_rabies::NUMERIC                                                AS rabies_per_year,
    ROUND(pandemic.avg_cases_per_period(n.number_rabies::NUMERIC, 12), 2)   AS avg_per_month,
    ROUND(pandemic.avg_cases_per_period(n.number_rabies::NUMERIC, 4), 2)    AS avg_per_quarter,
    ROUND(pandemic.avg_cases_per_period(n.number_rabies::NUMERIC, 2), 2)    AS avg_per_half_year
FROM pandemic.infectious_cases_norm n
JOIN pandemic.entities e ON e.id = n.entity_id
WHERE n.number_rabies ~ '^-?[0-9]+(\.[0-9]+)?$'
ORDER BY e.entity, n.year
LIMIT 20;
""")

## Підсумок

- Створено схему `pandemic`, дані з `infectious_cases.csv` імпортовано.
- Таблицю нормалізовано до 3НФ: `entities` (унікальні Entity і Code) та `infectious_cases_norm` (показники з `entity_id`), кількість рядків збігається з сирою таблицею.
- Для `Number_rabies` пораховано середнє, мінімум, максимум і суму, топ-10 за середнім.
- Різницю в роках побудовано вбудованими функціями `MAKE_DATE`, `CURRENT_DATE`, `AGE`.
- Створено дві функції (`year_diff`, `avg_cases_per_period`) і використано в запитах з дільниками 12, 4 і 2.